<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES.
All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Reindex with a new embedding model

This notebook shows how to reindex a document corpus with a different embedding model in NeMo Retriever Library (NRL) without extracting the documents again. Extraction (page rendering, layout detection, and OCR) does not depend on the embedding model, so it runs once and saves the extracted rows to Parquet. Each reindex reads the Parquet files, embeds the rows, and writes a new LanceDB index:

```text
Pipeline A, once:           PDFs → split → extract → page elements → OCR → rows → Parquet
Pipeline B, once per model: Parquet → embed → LanceDB → questions → Recall and nDCG
```

The corpus is [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr). The notebook reindexes it with **Gemma 300M** (`google/embeddinggemma-300m`, the baseline), **Nemotron 3 Embed 1B**, and **Nemotron 3 Embed 8B**, and evaluates each index with Recall and nDCG. It also records the GPU time of each stage and converts it to an estimated GPU cost.

Use a **Python 3.12** kernel. Edit the configuration cell, then choose **Run All Cells**.

## 1. Prepare the environment

Start with a Python 3.12 environment following the
[installation guide](https://github.com/NVIDIA/NeMo-Retriever/tree/main/nemo_retriever#setup-your-environment).
The measurements are meant for one **NVIDIA RTX PRO 6000 Blackwell Server Edition**.
Other data center GPUs with 48 GB or more also work, but without Blackwell the 1B model runs in BF16 and the timings differ.

For a new environment, set `INSTALL_DEPENDENCIES=True` and run the next cell once. It installs NRL from this repository,
because the notebook uses operators that are newer than the PyPI release, and replaces Torch and Torchvision with the
CUDA 13 builds specified in the guide. Restart the kernel afterward and switch installation off.

Two more requirements:

- **Gemma access.** Gemma 300M is gated. Accept the license on the [model page](https://huggingface.co/google/embeddinggemma-300m), then log in with `hf auth login` or set `HF_TOKEN`.
- **CUDA toolkit for NVFP4.** On first use, FlashInfer compiles and tunes the NVFP4 kernels for the 1B model and caches them. Start Jupyter from a shell where the environment is activated and `CUDA_HOME` points to a CUDA 13 toolkit, for example `export CUDA_HOME=/usr/local/cuda-13`, so `ninja` and `nvcc` are on the path.

The run takes about 8 minutes on one RTX PRO 6000 once the models (about 25 GB) are cached.

In [ ]:
import sys
from pathlib import Path

INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    nrl_source = Path("../nemo_retriever").resolve()
    !uv pip install --python "{sys.executable}" "{nrl_source}[local]" matplotlib ipywidgets
    !uv pip uninstall --python "{sys.executable}" torch torchvision
    !uv pip install --python "{sys.executable}" torch==2.11.0 torchvision==0.26.0 --index-url https://download.pytorch.org/whl/cu130

## 2. Configure the run

`MODELS` lists the embedders to compare; the first one is your current model and the baseline for every comparison.
`nvidia/nemotron-3-embed-1b` is the NRL default embedder.

Data, Parquet files, and indexes are created in `reindex_cost_workdir` under the current working directory.
Each run extracts the corpus again and rebuilds every index.

In [ ]:
import os

# Pin the notebook to one GPU before any CUDA library is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("LANCE_LOG", "error")

MODELS = {
    "Gemma 300M": "google/embeddinggemma-300m",
    "Nemotron 3 Embed 1B": "nvidia/nemotron-3-embed-1b",
    "Nemotron 3 Embed 8B": "nvidia/Nemotron-3-Embed-8B-BF16",
}
BASELINE = next(iter(MODELS))

DATASET = "vidore_v3_hr"
QUERY_LANGUAGE = "english"
TOP_K = 10
KS = (1, 5, 10)

WORK_DIR = Path("reindex_cost_workdir").resolve()
PDF_DIR = WORK_DIR / "pdfs"
PARQUET_DIR = WORK_DIR / "extracted"
LANCEDB_URI = str(WORK_DIR / "lancedb")
MODEL_LOG = WORK_DIR / "model_output.log"

NRL resolves each model name to a concrete checkpoint. On Blackwell GPUs, Nemotron 3 Embed 1B runs its **NVFP4** checkpoint.
Nemotron 3 Embed 8B and Gemma 300M are published only in BF16, so they run in BF16 on every GPU.

In [ ]:
import pandas as pd
from IPython.display import display

from nemo_retriever.models import resolve_local_embed_model
from nemo_retriever.models.embed_model_spec import resolve_embed_model_spec

checkpoints = {}
for label, model_name in MODELS.items():
    checkpoint = resolve_local_embed_model(model_name, backend="vllm")
    checkpoints[label] = {
        "checkpoint": checkpoint,
        "weights": resolve_embed_model_spec(checkpoint).quantization or "BF16",
    }
display(pd.DataFrame(checkpoints).T)

if not resolve_local_embed_model("nvidia/nemotron-3-embed-1b", backend="vllm").endswith("NVFP4"):
    print(
        "This GPU has no native NVFP4 support, so the 1B model runs in BF16. "
        "Timings will not match the RTX PRO 6000 results in the summary."
    )

The helpers below keep notebook output brief and time every graph stage.

- `quiet_model_output` sends model-server output (vLLM workers, progress bars) to `MODEL_LOG`. Failures still raise exceptions and point to the log.
- `run_graph_timed` runs a graph in-process, like `InprocessExecutor`, and records the wall-clock time of each stage. For the embedding stage it embeds one row first, so model startup is timed apart from the work that grows with your corpus.

In [ ]:
import contextlib
import logging
import time

from nemo_retriever.graph import UDFOperator

# Keep notebook output brief; failures still raise exceptions.
logging.disable(logging.WARNING)


@contextlib.contextmanager
def quiet_model_output():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    sys.stdout.flush()
    sys.stderr.flush()
    saved = [os.dup(1), os.dup(2)]
    failed = False
    with open(MODEL_LOG, "a", buffering=1) as log:
        os.dup2(log.fileno(), 1)
        os.dup2(log.fileno(), 2)
        try:
            with contextlib.redirect_stdout(log), contextlib.redirect_stderr(log):
                yield
        except Exception:
            failed = True
            raise
        finally:
            os.dup2(saved[0], 1)
            os.dup2(saved[1], 2)
            for fd in saved:
                os.close(fd)
            if failed:
                print(f"Model output, including the root cause, is in {MODEL_LOG}")


def _stage_name(node):
    return node.operator.name if isinstance(node.operator, UDFOperator) else node.name


def graph_stages(graph):
    names, node = [], graph.roots[0]
    while node is not None:
        names.append(_stage_name(node))
        node = node.children[0] if node.children else None
    return names


def run_graph_timed(graph, data, *, warmup_stage=None):
    start = time.perf_counter()
    resolved = graph.resolve_for_local_execution()
    timings = [{"stage": "graph setup", "seconds": time.perf_counter() - start, "rows_out": None}]
    node = resolved.roots[0]
    while node is not None:
        name = _stage_name(node)
        if name == warmup_stage:
            start = time.perf_counter()
            node.operator.run(data[data["text"].fillna("").str.strip() != ""].head(1))
            timings.append({"stage": f"{name} (model startup)", "seconds": time.perf_counter() - start, "rows_out": 1})
        start = time.perf_counter()
        data = node.operator.run(data)
        timings.append({"stage": name, "seconds": time.perf_counter() - start, "rows_out": len(data)})
        node = node.children[0] if node.children else None
    return data, pd.DataFrame(timings)

## 3. Prepare the dataset

Load [ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr): 14 EU reports about employment and labor markets
(1,110 pages) and English questions labeled with the pages that answer them. The BEIR helper loads the questions and
relevance labels and maps each labeled page to a `"<document>_<page>"` ID, the same `pdf_page` ID NRL attaches to retrieval hits.

The dataset ships each page as an image. Regroup the pages into one PDF per report, which is what a document store
would contain. The rebuilt PDFs have no text layer, so Pipeline A must OCR every page. Existing PDFs are reused.

In [ ]:
from collections import defaultdict

from datasets import load_dataset

from nemo_retriever.tools.recall.beir import load_beir_dataset

corpus = load_dataset(f"vidore/{DATASET}", "corpus", split="test")
eval_set = load_beir_dataset("vidore_hf", dataset_name=DATASET, query_language=QUERY_LANGUAGE, doc_id_field="pdf_page")
query_ids, queries, qrels = eval_set.query_ids, eval_set.queries, eval_set.qrels

pages_by_doc = defaultdict(list)
for row_index, (doc_id, page) in enumerate(zip(corpus["doc_id"], corpus["page_number_in_doc"])):
    pages_by_doc[doc_id].append((page, row_index))

PDF_DIR.mkdir(parents=True, exist_ok=True)
for doc_id, pages in pages_by_doc.items():
    pdf_path = PDF_DIR / f"{doc_id}.pdf"
    if not pdf_path.exists():
        images = [corpus[row_index]["image"].convert("RGB") for _, row_index in sorted(pages)]
        images[0].save(pdf_path, "PDF", save_all=True, append_images=images[1:], resolution=144)

pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
assert pdf_paths and queries, "The corpus and questions must not be empty."
print(f"{len(pdf_paths)} documents · {len(corpus):,} pages · {len(queries):,} questions")
print(f"Example question: {queries[0]}")

## 4. Extract once to Parquet

Pipeline A is the NRL default PDF extraction graph from `build_graph`, the same graph that
`create_ingestor().files(...).extract(method="pdfium_hybrid")` builds. **PDFium hybrid** reads native PDF text where it
exists and OCRs pages that have none. Two stages follow:

- `ExplodeContentActor` splits each page into one row per text block, table, chart, and infographic: the rows an embedder turns into vectors.
- `ParquetWriterOperator` writes the rows to `PARQUET_DIR`, without the base64 page images that text embedding does not use.

In [ ]:
import shutil

from nemo_retriever.common.params import ExtractParams
from nemo_retriever.graph import ParquetWriterOperator
from nemo_retriever.graph.ingestor_runtime import build_graph
from nemo_retriever.operators import ExplodeContentActor

shutil.rmtree(PARQUET_DIR, ignore_errors=True)
extract_graph = (
    build_graph(extraction_mode="pdf", extract_params=ExtractParams(method="pdfium_hybrid"))
    >> ExplodeContentActor(modality="text")
    >> ParquetWriterOperator(output_dir=str(PARQUET_DIR), exclude_columns=["page_image", "images"])
)
print(" → ".join(graph_stages(extract_graph)))

print("Extracting pages…")
pdf_input = pd.DataFrame([{"bytes": p.read_bytes(), "path": str(p)} for p in pdf_paths])
with quiet_model_output():
    _, extract_timings = run_graph_timed(extract_graph, pdf_input)
assert any(PARQUET_DIR.glob("*.parquet")), "Extraction wrote no Parquet files."
display(extract_timings.style.format({"seconds": "{:.1f}", "rows_out": "{:,.0f}"}, na_rep=""))

## 5. Describe the extracted data

These numbers describe the input to every reindex. Compare them with your own corpus: embedding time grows with the
number of rows and the amount of text, not with the number of PDF files. Rows without text, such as blank pages, are not embedded.

In [ ]:
from nemo_retriever.graph import ParquetReaderOperator

extracted = ParquetReaderOperator().run(str(PARQUET_DIR))
has_text = extracted["text"].fillna("").str.strip() != ""
chars = extracted.loc[has_text, "text"].str.len()
N_PAGES = sum(len(pages_by_doc[p.stem]) for p in pdf_paths)
extract_seconds = extract_timings["seconds"].sum()

dataset_stats = pd.Series(
    {
        "documents": len(pdf_paths),
        "pages": N_PAGES,
        "extracted rows": len(extracted),
        "rows with text to embed": has_text.sum(),
        "extracted text (million characters)": chars.sum() / 1e6,
        "characters per page, average": chars.sum() / N_PAGES,
        "characters per row, average": chars.mean(),
        "characters per row, median": chars.median(),
        "characters per row, 95th percentile": chars.quantile(0.95),
        "Parquet size (MB)": sum(f.stat().st_size for f in PARQUET_DIR.glob("*.parquet")) / 1e6,
        "extraction time (minutes)": extract_seconds / 60,
    },
    name=DATASET,
)
display(
    dataset_stats.to_frame().style.format(lambda v: f"{v:,.0f}" if float(v).is_integer() or v > 100 else f"{v:,.2f}")
)

by_type = (
    extracted[has_text]
    .assign(characters=chars)
    .groupby("_content_type")["characters"]
    .agg(rows="count", average_characters="mean", total_characters="sum")
    .sort_values("rows", ascending=False)
)
display(by_type.style.format("{:,.0f}"))

## 6. Reindex with each model

Pipeline B is three stages, and none of them touches a PDF:

1. `ParquetReaderOperator` loads the rows written in step 4.
2. `_BatchEmbedActor` embeds the text with the chosen model: the embedding stage of the NRL default ingestion graph.
3. `IngestVdbOperator` writes one LanceDB table per model and records the model on it. NRL's `Retriever` refuses to query that table with any other model.

Each model's GPU memory is released before the next model loads. Only the embedding stage depends on the model.

In [ ]:
import gc

import lancedb

from nemo_retriever.common.params import EmbedParams
from nemo_retriever.operators import _BatchEmbedActor
from nemo_retriever.operators.vdb import IngestVdbOperator


def table_name(label):
    return label.lower().replace(" ", "_")


def build_reindex_graph(model_name, table):
    return (
        ParquetReaderOperator()
        >> _BatchEmbedActor(params=EmbedParams(model_name=model_name))
        >> IngestVdbOperator(
            vdb_op="lancedb",
            vdb_kwargs={
                "uri": LANCEDB_URI,
                "table_name": table,
                "overwrite": True,
                "embedding_model_name": model_name,
                "vector_dim": None,
            },
        )
    )


print(" → ".join(graph_stages(build_reindex_graph(MODELS[BASELINE], table_name(BASELINE)))))

results = {}
for label, model_name in MODELS.items():
    print(f"Reindexing with {label}…")
    graph = build_reindex_graph(model_name, table_name(label))
    with quiet_model_output():
        _, timings = run_graph_timed(graph, str(PARQUET_DIR), warmup_stage="_BatchEmbedActor")
    del graph
    gc.collect()
    rows_indexed = lancedb.connect(LANCEDB_URI).open_table(table_name(label)).count_rows()
    assert rows_indexed > 0, f"The {label} index is empty; see {MODEL_LOG}."
    results[label] = {"timings": timings, "rows_indexed": rows_indexed}

stage_seconds = pd.concat({label: r["timings"].set_index("stage")["seconds"] for label, r in results.items()}, axis=1)
display(stage_seconds.style.format("{:.1f}"))

## 7. Retrieve and evaluate

Embed each question with the model that built the index and retrieve the top ten rows. The BEIR helper maps the hits to
page IDs and converts them into the format used for evaluation.

**Recall@k** measures the fraction of relevant pages found in the first k results.
**nDCG@k** rewards putting the most relevant pages first, using the labels' relevance grades and a logarithmic rank discount.
Higher is better for both metrics. The evaluation is timed too: loading the question embedder, measured with one warm-up
question, then embedding every question and searching the index.

In [ ]:
from nemo_retriever.graph.retriever import Retriever
from nemo_retriever.tools.recall.beir import build_beir_run_from_hits, compute_beir_metrics

METRIC_NAMES = [f"recall@{k}" for k in KS] + [f"ndcg@{k}" for k in KS]

for label, model_name in MODELS.items():
    print(f"Evaluating {label}…")
    with quiet_model_output():
        retriever = Retriever(
            vdb_kwargs={"uri": LANCEDB_URI, "table_name": table_name(label)},
            embed_kwargs={"model_name": model_name},
            top_k=TOP_K,
        )
        start = time.perf_counter()
        retriever.queries(queries[:1])
        hits = retriever.queries(queries)
        eval_seconds = time.perf_counter() - start
        del retriever
        gc.collect()
    run = build_beir_run_from_hits(query_ids, hits, doc_id_field="pdf_page")
    assert run, f"No pages were retrieved for {label}."
    metrics = compute_beir_metrics(qrels, run, ks=KS)
    results[label].update(metrics={m: metrics[m] for m in METRIC_NAMES}, eval_seconds=eval_seconds, run=run)

quality = pd.DataFrame(
    {
        (label, name): [r["metrics"][f"{metric}@{k}"] for k in KS]
        for label, r in results.items()
        for name, metric in (("Recall", "recall"), ("nDCG", "ndcg"))
    },
    index=pd.Index(KS, name="k"),
)
display(quality.style.format("{:.3f}"))

## 8. Compare cost and quality

Costs use the Linux on-demand price of `GPU_INSTANCE`, the AWS EC2 instance with one RTX PRO 6000 Blackwell Server
Edition. Prices change and differ by region, so check the
[EC2 On-Demand pricing page](https://aws.amazon.com/ec2/pricing/on-demand/) and set `GPU_PRICE_USD_PER_HOUR`.

The cost of a model is the GPU time of the reindex plus the evaluation, rounded up to whole hours and multiplied by
the on-demand price. At 1,110 pages every model finishes well inside one hour, so the billed cost here is the same for
all three; step 9 scales the measured time to a larger corpus, where the difference shows. `rows indexed` can differ
between models because each embedder splits text longer than its input limit into several rows.

In [ ]:
import math

GPU_INSTANCE = "g7e.2xlarge"  # AWS EC2, one NVIDIA RTX PRO 6000 Blackwell Server Edition (96 GB)
GPU_PRICE_USD_PER_HOUR = 3.36312  # Linux on-demand, AWS Pricing Calculator, October 2026
assert GPU_PRICE_USD_PER_HOUR > 0


def billed_hours(seconds):
    return math.ceil(seconds / 3600)


def gpu_cost_usd(seconds):
    return billed_hours(seconds) * GPU_PRICE_USD_PER_HOUR


summary_df = pd.DataFrame(
    {
        label: {
            "rows indexed": r["rows_indexed"],
            "reindex time (min)": r["timings"]["seconds"].sum() / 60,
            "evaluation time (min)": r["eval_seconds"] / 60,
            "GPU-hours": (r["timings"]["seconds"].sum() + r["eval_seconds"]) / 3600,
            "billed GPU-hours": billed_hours(r["timings"]["seconds"].sum() + r["eval_seconds"]),
            "GPU cost (USD)": gpu_cost_usd(r["timings"]["seconds"].sum() + r["eval_seconds"]),
            "ndcg@10": r["metrics"]["ndcg@10"],
            "recall@10": r["metrics"]["recall@10"],
        }
        for label, r in results.items()
    }
).T.astype(float)
print(f"{GPU_INSTANCE} at ${GPU_PRICE_USD_PER_HOUR:.2f} per hour, {len(queries)} questions")
display(
    summary_df.style.format(
        {
            "rows indexed": "{:,.0f}",
            "reindex time (min)": "{:.1f}",
            "evaluation time (min)": "{:.1f}",
            "GPU-hours": "{:.2f}",
            "billed GPU-hours": "{:.0f}",
            "GPU cost (USD)": "${:,.2f}",
            "ndcg@10": "{:.3f}",
            "recall@10": "{:.3f}",
        }
    )
)

baseline_metrics = pd.Series(results[BASELINE]["metrics"])
gains = pd.DataFrame(
    {
        f"{label} vs {BASELINE}": {
            **{f"{m} change": r["metrics"][m] - baseline_metrics[m] for m in METRIC_NAMES},
            "ndcg@10 relative change": r["metrics"]["ndcg@10"] / baseline_metrics["ndcg@10"] - 1,
        }
        for label, r in results.items()
        if label != BASELINE
    }
)
display(gains.style.format("{:+.3f}").format("{:+.1%}", subset=pd.IndexSlice[["ndcg@10 relative change"], :]))

Averages hide how individual questions move. For each question, compare nDCG@10 with the baseline:

In [ ]:
def per_query_ndcg(run):
    return pd.Series({q: compute_beir_metrics({q: qrels[q]}, run, ks=(10,))["ndcg@10"] for q in query_ids})


baseline_ndcg = per_query_ndcg(results[BASELINE]["run"])
movement = {}
for label, r in results.items():
    if label != BASELINE:
        delta = per_query_ndcg(r["run"]) - baseline_ndcg
        movement[label] = {
            "improved": (delta > 0).sum(),
            "unchanged": (delta == 0).sum(),
            "regressed": (delta < 0).sum(),
        }
display(pd.DataFrame(movement))

## 9. Scale the cost to your corpus

Set `YOUR_CORPUS_PAGES` to the size of your corpus. The estimate keeps model startup as a fixed cost and scales the rest
of the measured reindex time by the page count. It assumes pages that carry about as much text as this corpus (step 5);
if yours carry twice the text per page, use twice the page count. GPU-hours are the same on one GPU or many: eight
GPUs finish in about an eighth of the wall-clock time for about the same cost.

Extraction already ran once, so it is not part of any reindex. The last row shows what re-extracting the same corpus
would cost, for comparison.

In [ ]:
YOUR_CORPUS_PAGES = 1_000_000

scale = YOUR_CORPUS_PAGES / N_PAGES


def scaled_seconds(timings):
    fixed = timings.loc[timings["stage"].str.endswith("(model startup)"), "seconds"].sum()
    return fixed + (timings["seconds"].sum() - fixed) * scale


rows = {}
for label, r in results.items():
    seconds = scaled_seconds(r["timings"])
    rows[label] = {
        "pages per second (pps)": YOUR_CORPUS_PAGES / seconds,
        "GPU-hours": seconds / 3600,
        "billed GPU-hours": billed_hours(seconds),
        "GPU cost (USD)": gpu_cost_usd(seconds),
        "ndcg@10": r["metrics"]["ndcg@10"],
        "recall@10": r["metrics"]["recall@10"],
    }
at_scale = pd.DataFrame(rows).T.astype(float)
at_scale["extra cost vs baseline (USD)"] = at_scale["GPU cost (USD)"] - at_scale.loc[BASELINE, "GPU cost (USD)"]
at_scale["ndcg@10 gain vs baseline"] = at_scale["ndcg@10"] - at_scale.loc[BASELINE, "ndcg@10"]
at_scale["recall@10 gain vs baseline"] = at_scale["recall@10"] - at_scale.loc[BASELINE, "recall@10"]

extract_at_scale = extract_seconds * scale
at_scale.loc[
    "Re-extraction (not needed)", ["pages per second (pps)", "GPU-hours", "billed GPU-hours", "GPU cost (USD)"]
] = [
    YOUR_CORPUS_PAGES / extract_at_scale,
    extract_at_scale / 3600,
    billed_hours(extract_at_scale),
    gpu_cost_usd(extract_at_scale),
]
print(f"Estimate for {YOUR_CORPUS_PAGES:,} pages on one {GPU_INSTANCE} at ${GPU_PRICE_USD_PER_HOUR:.2f} per hour")
display(
    at_scale.style.format(
        {
            "pages per second (pps)": "{:,.1f}",
            "GPU-hours": "{:.1f}",
            "billed GPU-hours": "{:.0f}",
            "GPU cost (USD)": "${:,.2f}",
            "ndcg@10": "{:.3f}",
            "recall@10": "{:.3f}",
            "extra cost vs baseline (USD)": lambda v: f"{'-' if v < 0 else '+'}${abs(v):,.2f}",
            "ndcg@10 gain vs baseline": "{:+.3f}",
            "recall@10 gain vs baseline": "{:+.3f}",
        },
        na_rep="—",
    )
)

The chart puts the estimated cost next to the quality each model delivers.

In [ ]:
import matplotlib.pyplot as plt

models = list(MODELS)
fig, (cost_ax, quality_ax) = plt.subplots(1, 2, figsize=(11, 3.5))
cost_ax.barh(models, at_scale.loc[models, "GPU cost (USD)"], color="#76b900")
cost_ax.set_xlabel("USD")
cost_ax.set_title(f"Reindex cost, {YOUR_CORPUS_PAGES:,} pages")
cost_ax.invert_yaxis()
quality_ax.barh(models, at_scale.loc[models, "ndcg@10"], color="#5d8aa8")
quality_ax.set_xlabel("nDCG@10")
quality_ax.set_title(f"Retrieval quality, {len(queries)} questions")
quality_ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

- Pipeline A extracted the PDFs once and saved the rows to Parquet. Reindexing did not repeat rendering, layout detection, or OCR.
- Pipeline B built one index per embedding model with three stages: `ParquetReaderOperator`, `_BatchEmbedActor`, and `IngestVdbOperator`.
- Each index was evaluated on the same questions and relevance labels with Recall and nDCG at 1, 5, and 10.
- Each model was timed on the same GPU. Step 9 converts the measured time into GPU-hours and an estimated cost, billed in whole hours at the on-demand price, for the corpus size you set.

### Results from the validation run

One RTX PRO 6000 Blackwell Server Edition, `g7e.2xlarge` at $3.36312 per hour, October 2026. Pages per second and cost per million pages come from step 9: measured throughput, billed hours rounded up. Changes are relative to Gemma 300M.

| Model (weights) | Pages per second (pps) | Cost per 1M pages | nDCG@10 | Recall@10 |
|---|---|---|---|---|
| Gemma 300M (BF16) | 148 | $6.73 | 0.529 | 0.564 |
| Nemotron 3 Embed 1B (NVFP4) | 120 | $10.09 (+$3.36) | 0.594 (+0.065, +12%) | 0.616 (+0.052) |
| Nemotron 3 Embed 8B (BF16) | 22 | $43.72 (+$36.99) | 0.642 (+0.113, +21%) | 0.658 (+0.094) |

Re-extracting the same million pages would take 52 billed GPU-hours, about $175 at the same price. Scores vary slightly between runs.

To evaluate a full RAG pipeline on your own data, refer to [NeMo Retriever RAGAS evaluation](nrl_ragas.ipynb).